# Milestone 3: Preprocessing & Distributed Model

**Dataset:** FineWeb-Edu 10BT Sample (14 Parquet files)  
**Platform:** SDSC Expanse — 32 cores total (1 driver + 31 executors)  
**Task:** 5-class educational quality scoring (`int_score` 0–4)

---

## Setup & Imports

In [3]:
!pip install -qq pyspark

import os
import requests
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.ml.feature import (
    RegexTokenizer, 
    Word2Vec, 
    VectorAssembler, 
    StandardScaler, 
    StopWordsRemover,
    Imputer,
    MinMaxScaler
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

## Data Acquisition

Download 14 Parquet files from the FineWeb-Edu 10BT HuggingFace dataset into `../data/`. Files are skipped if already present.

In [4]:
# Set the target directory to '../data' relative to the notebook
data_dir = "../data/raw"
os.makedirs(data_dir, exist_ok=True)

# Base URL for the 10BT Sample
base_url = "https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/main/sample/10BT"

# Loop through files 000 to 013
for i in range(14):
    filename = f"{i:03d}_00000.parquet"
    save_path = os.path.join(data_dir, filename)
    
    # Only download if missing
    if not os.path.exists(save_path):
        print(f"Downloading {filename}...")
        !wget -q -O {save_path} {base_url}/{filename}
    else:
        print(f"Skipping {filename} (Already exists)")

print("All 14 sample files downloaded.")

All 14 sample files downloaded.


## Spark Session & Cluster Verification

Start a Spark session on SDSC Expanse with 31 executors, then verify distributed resources via the Spark REST API.

In [3]:
# Driver: 2g (coordinator only, does not process data)
# Executor instances: Total Cores - 1 = 32 - 1 = 31
# Memory per executor: (128g - 2g) / (31 × 1.1 overhead) ≈ 3g
# Actual usage: 31 × (3g × 1.1) + 2g ≈ 104g — safely within 128GB
spark = SparkSession.builder \
    .appName("Milestone3_Complete_Pipeline") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.instances", "31") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "3g") \
    .getOrCreate()

spark

In [4]:
# Get the active Spark Context and URL
sc = spark.sparkContext
url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"

# Fetch the executor data from the API
response = requests.get(url)
executors = response.json()

# Format into a readable DataFrame
df = pd.DataFrame(executors)[['id', 'totalCores', 'maxMemory', 'activeTasks', 'isActive']]
df['maxMemory_GB'] = (df['maxMemory'] / (1024**3)).round(2)
df

,id,totalCores,maxMemory,activeTasks,isActive,maxMemory_GB
0,driver,32,1099746508,0,True,1.02


## Load & Explore Data

Read all 14 Parquet files into a single Spark DataFrame, inspect the schema, and preview sample rows.

In [5]:
# Read all parquet files from the '../data' directory into a single DataFrame
df = spark.read.parquet("../data")

In [6]:
# Show the schema of the DataFrame to verify it loaded correctly
df.printSchema()

root
 |-- text: string (nullable = true)
 |-- id: string (nullable = true)
 |-- dump: string (nullable = true)
 |-- url: string (nullable = true)
 |-- file_path: string (nullable = true)
 |-- language: string (nullable = true)
 |-- language_score: double (nullable = true)
 |-- token_count: long (nullable = true)
 |-- score: double (nullable = true)
 |-- int_score: long (nullable = true)



In [7]:
# Shows 10 rows, truncating every column to a maximum of 10 characters
df.show(10, truncate=10)

+----------+----------+----------+----------+----------+--------+--------------+-----------+--------+---------+
|      text|        id|      dump|       url| file_path|language|language_score|token_count|   score|int_score|
+----------+----------+----------+----------+----------+--------+--------------+-----------+--------+---------+
|Whether...|<urn:uu...|CC-MAIN...|http://...|s3://co...|      en|    0.93640...|        619|2.953125|        3|
||Skip N...|<urn:uu...|CC-MAIN...|http://...|s3://co...|      en|    0.89046...|        189|2.546875|        3|
|PRINT T...|<urn:uu...|CC-MAIN...|http://...|s3://co...|      en|    0.94871...|        408|3.171875|        3|
|BRISBAN...|<urn:uu...|CC-MAIN...|https:/...|s3://co...|      en|    0.94921...|        621| 2.84375|        3|
|This th...|<urn:uu...|CC-MAIN...|https:/...|s3://co...|      en|    0.93207...|        566|2.515625|        3|
|The fur...|<urn:uu...|CC-MAIN...|http://...|s3://co...|      en|    0.97275...|        636|3.984375|   

# PREPROCESSING PIPELINE 

### Data Cleaning & Feature Engineering

In [8]:
# Filtering Invalid Documents
df = df.filter(
    (F.col("text").isNotNull()) &
    (F.length(F.col("text")) > 200) &
    (F.col("token_count") >= 50)
)

In [9]:
# Sample 20% of the CLEAN data for fast development
df = df.sample(withReplacement=False, fraction=0.20, seed=42)

# Save this clean sample to disk so Spark doesn't have to recalculate it
spark.sparkContext.setCheckpointDir("../data/checkpoints")
df = df.checkpoint()

print(f"Cleaned and sampled rows: {df.count()}")

Cleaned and sampled rows: 1935821


In [10]:
# Feature Engineering using Spark SQL functions (regexp_extract, length, when, otherwise)
df = df.withColumn(
    "domain",
    F.regexp_extract(F.col("url"), r"https?://([^/]+)", 1)
)
df = df.withColumn(
    "length_bucket",
    F.when(F.col("token_count") < 500, "short")
     .when(F.col("token_count") < 2000, "medium")
     .otherwise("long")
)

In [11]:
# Scaling token_count, score, language_score using MinMaxScaler to the [0,1] range
num_assembler = VectorAssembler(
    inputCols=["token_count", "score", "language_score"],
    outputCol="numeric_features"
)

df = num_assembler.transform(df)

minmax_scaler = MinMaxScaler(
    inputCol="numeric_features",
    outputCol="scaled_features"
)

minmax_scaler_model = minmax_scaler.fit(df)
df = minmax_scaler_model.transform(df)

In [12]:
#  Apply dropDuplicates(["id"]) to remove duplicate documents
df = df.dropDuplicates(["id"])

In [13]:
# Defining the target variable (quality)
df = df.withColumn("label", F.col("int_score").cast(IntegerType()))

In [14]:
# Show the distribution of the target variable (int_score) to check for class imbalance
df.groupBy("int_score").count().orderBy("int_score").show()

+---------+-------+
|int_score|  count|
+---------+-------+
|        3|1678010|
|        4| 256326|
|        5|   1485|
+---------+-------+



### ML Pipeline Transformer Definitions

Define the Spark MLlib transformers for text processing and feature assembly. These stages will be composed into a single `Pipeline` in the modeling section.

In [ ]:
# Split raw text into lowercase word tokens, removing punctuation
tokenizer = RegexTokenizer(inputCol="text", outputCol="raw_words", pattern="\\W+", toLowercase=True)

# Remove common English stop words to reduce noise and vocabulary size.
remover = StopWordsRemover(inputCol="raw_words", outputCol="filtered_words")

# Map filtered words to 10-dimensional dense vectors. Words appearing fewer than 500 times are ignored.
word2vec = Word2Vec(vectorSize=10, minCount=50, inputCol="filtered_words", outputCol="text_features")

# Impute missing token_count values using the mean strategy, creating a new column token_count_imputed
imputer = Imputer(inputCols=["token_count"], outputCols=["token_count_imputed"], strategy="mean")

# Assemble the imputed token_count into a vector for scaling
vec_assembler_num = VectorAssembler(inputCols=["token_count_imputed"], outputCol="token_vec")
scaler = StandardScaler(inputCol="token_vec", outputCol="token_scaled", withStd=True, withMean=True)

# Combine text features and scaled token_count into a single feature vector for modeling
final_assembler = VectorAssembler(
    inputCols=["text_features", "token_scaled"], 
    outputCol="features"
)

# DISTRIBUTED MODEL

### Random Forest (baseline)

`numTrees=5`, `maxDepth=2` — lightweight baseline to establish a performance floor.

In [16]:
# Initialize RandomForestClassifier
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=5, maxDepth=2)

# build pipeline (label already created before pipeline via SOP preprocessing)
pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    word2vec,
    imputer,         
    vec_assembler_num,
    scaler,
    final_assembler,
    rf
])

### Data Split & Stratified Sampling

To limit compute cost, we first take a **1% flat sample** of the full dataset, then apply **stratified sampling** on that subset to balance the class distribution before splitting 60/20/20:

- **Scores 0, 1, 2** — keep 100% (rare minority classes)
- **Score 4** — keep 30% (moderately common)
- **Score 3** — keep 5% (dominant class, heavily undersampled)

In [17]:
# Stratified sampling on the subset to address class imbalance:
#   int_score 0, 1, 2 -> keep 100% (rare minority classes)
#   int_score 3       -> keep 5%   (dominant class ~86.7% of data)
#   int_score 4       -> keep 30%  (moderately common)
fractions = {0: 1.0, 1: 1.0, 2: 1.0, 3: 0.05, 4: 0.30}
df_sampled = df.sampleBy("int_score", fractions=fractions, seed=42)

# Break the execution graph AGAIN after the heavy network shuffle
df_sampled = df_sampled.checkpoint()

# Show resulting class distribution after sampling (Now 100% safe to run!)
print("Class distribution after stratified sampling:")
df_sampled.groupBy("int_score").count().orderBy("int_score").show()

# Split into Train / Validation / Test (60 / 20 / 20)
train_data, val_data, test_data = df_sampled.randomSplit([0.6, 0.2, 0.2], seed=42)

# This will execute quickly and safely
print(f"Train: {train_data.count()}  Val: {val_data.count()}  Test: {test_data.count()}")

Class distribution after stratified sampling:
+---------+-----+
|int_score|count|
+---------+-----+
|        3|83826|
|        4|76394|
+---------+-----+

Train: 96141  Val: 32063  Test: 32016


### Train Model 1

In [18]:
%%time
print("Training Distributed Model...")
model = pipeline.fit(train_data)
print("Training Complete.")

Training Distributed Model...
Training Complete.
CPU times: user 46.4 ms, sys: 17.3 ms, total: 63.7 ms
Wall time: 13min 15s


# FITTING ANALYSIS & EVALUATION

In [19]:
%%time
# compare training vs validation vs test error
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)

train_preds = model.transform(train_data)
val_preds   = model.transform(val_data)
test_preds  = model.transform(test_data)

train_acc = evaluator.evaluate(train_preds)
val_acc   = evaluator.evaluate(val_preds)
test_acc  = evaluator.evaluate(test_preds)

print(f"Model 1 — RF (numTrees=5, maxDepth=2)")
print(f"  Train Accuracy : {train_acc:.4f}")
print(f"  Val   Accuracy : {val_acc:.4f}")
print(f"  Test  Accuracy : {test_acc:.4f}")

Model 1 — RF (numTrees=5, maxDepth=2)
  Train Accuracy : 0.6219
  Val   Accuracy : 0.6197
  Test  Accuracy : 0.6189
CPU times: user 53.7 ms, sys: 15.7 ms, total: 69.3 ms
Wall time: 6.04 s


In [20]:
print("\n TRAIN SET Predictions (Model 1)")
train_preds.select("int_score", "prediction").show(10, truncate=50)

print("\n VALIDATION SET Predictions (Model 1)")
val_preds.select("int_score", "prediction").show(10, truncate=50)

print("\n TEST SET Predictions (Model 1)")
test_preds.select("int_score", "prediction").show(10, truncate=50)


 TRAIN SET Predictions (Model 1)
+---------+----------+
|int_score|prediction|
+---------+----------+
|        4|       4.0|
|        4|       4.0|
|        4|       3.0|
|        3|       3.0|
|        4|       4.0|
|        3|       3.0|
|        3|       3.0|
|        4|       4.0|
|        4|       4.0|
|        4|       3.0|
+---------+----------+
only showing top 10 rows


 VALIDATION SET Predictions (Model 1)
+---------+----------+
|int_score|prediction|
+---------+----------+
|        4|       3.0|
|        4|       4.0|
|        4|       3.0|
|        4|       3.0|
|        3|       3.0|
|        3|       3.0|
|        3|       3.0|
|        3|       3.0|
|        4|       4.0|
|        4|       3.0|
+---------+----------+
only showing top 10 rows


 TEST SET Predictions (Model 1)
+---------+----------+
|int_score|prediction|
+---------+----------+
|        3|       3.0|
|        3|       4.0|
|        4|       4.0|
|        3|       3.0|
|        3|       3.0|
|        3|   

### Model 2 — Random Forest (deeper, more trees)

`numTrees=20`, `maxDepth=8` — tests how additional capacity affects the bias–variance trade-off.

In [21]:
rf2 = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=20, maxDepth=8)

pipeline2 = Pipeline(stages=[
    tokenizer,
    remover,
    word2vec,
    imputer,
    vec_assembler_num,
    scaler,
    final_assembler,
    rf2
])

In [22]:
%%time
print("Training Model 2 (numTrees=20, maxDepth=8)...")
model2 = pipeline2.fit(train_data)
print("Training Complete.")

Training Model 2 (numTrees=20, maxDepth=8)...
Training Complete.
CPU times: user 41.7 ms, sys: 14.2 ms, total: 55.8 ms
Wall time: 13min 16s


In [23]:
# Evaluate Model 2 across all three splits
train_preds2 = model2.transform(train_data)
val_preds2   = model2.transform(val_data)
test_preds2  = model2.transform(test_data)

train_acc2 = evaluator.evaluate(train_preds2)
val_acc2   = evaluator.evaluate(val_preds2)
test_acc2  = evaluator.evaluate(test_preds2)

print(f"Model 2 — RF (numTrees=20, maxDepth=8)")
print(f"  Train Accuracy : {train_acc2:.4f}")
print(f"  Val   Accuracy : {val_acc2:.4f}")
print(f"  Test  Accuracy : {test_acc2:.4f}")

print("\n--- Comparison ---")
print(f"{'Model':<35} {'Train':>7} {'Val':>7} {'Test':>7}")
print(f"{'RF (numTrees=5, maxDepth=2)':<35} {train_acc:>7.4f} {val_acc:>7.4f} {test_acc:>7.4f}")
print(f"{'RF (numTrees=20, maxDepth=8)':<35} {train_acc2:>7.4f} {val_acc2:>7.4f} {test_acc2:>7.4f}")

Model 2 — RF (numTrees=20, maxDepth=8)
  Train Accuracy : 0.6998
  Val   Accuracy : 0.6870
  Test  Accuracy : 0.6867

--- Comparison ---
Model                                 Train     Val    Test
RF (numTrees=5, maxDepth=2)          0.6219  0.6197  0.6189
RF (numTrees=20, maxDepth=8)         0.6998  0.6870  0.6867


## Fitting Analysis

**1. Where does the model sit on the underfitting / overfitting spectrum?**

After filtering and 20% sampling the dataset contains **1,935,821** rows. The raw `int_score` distribution is only **3 classes** (3, 4, 5) — not 0–4 as originally expected. Stratified sampling (fractions defined for keys 0–4) retains only classes **3** and **4** (int_score=5 is absent from the fractions dict and therefore dropped by `sampleBy`), yielding an effective **2-class** classification problem:

| int_score | After sampling |
|---|---|
| 3 | 83,826 |
| 4 | 76,394 |

**Train / Val / Test split: 96,141 / 32,063 / 32,016**

The majority-class baseline (always predicting class 3) is ≈ **52.3%**.

Actual accuracy results:

| Model | Train | Val | Test | Train–Test Gap |
|---|---|---|---|---|
| RF (numTrees=5, maxDepth=2) | 0.6219 | 0.6197 | 0.6189 | 0.003 |
| RF (numTrees=20, maxDepth=8) | 0.6998 | 0.6870 | 0.6867 | 0.013 |

**Model 1** has a train–test gap of only **0.3 pp** — virtually no overfitting, but the shallow trees (maxDepth=2) limit capacity and leave the model closer to the underfitting end. Test accuracy of 61.89% is only ~9.6 pp above the majority-class baseline.

**Model 2** shows a wider train–test gap of **1.31 pp**, indicating mild overfitting as the deeper, more numerous trees begin to capture training-specific noise. However, the gap is still manageable and the model delivers meaningfully better accuracy.

**2. Build at least one model with different hyperparameters and compare results**

Two Random Forest models were trained:
- **Model 1**: `numTrees=5`, `maxDepth=2` — very shallow trees; almost no overfitting but limited discriminative power
- **Model 2**: `numTrees=20`, `maxDepth=8` — deeper trees with more estimators; substantially higher model capacity

Model 2 outperforms Model 1 on every split (train, val, and test), gaining roughly **+7.8 pp** in test accuracy at the cost of a mildly wider generalization gap.

**3. Which model performs best and why?**

**Model 2** achieves the best test accuracy (**0.6867** vs. 0.6189). The increased depth allows each tree to learn higher-order feature interactions between Word2Vec embeddings and scaled token count, while 20 trees provide stronger ensemble variance reduction compared to just 5. The train–test gap (1.31 pp) is acceptable for this data scale and confirms that the accuracy gain reflects genuine generalization rather than pure memorization.

**4. What are the next models planned for Milestone 4 and why?**

- **Gradient Boosted Trees (GBTClassifier)**: Sequential boosting iteratively corrects residual errors from prior trees, which should close the remaining accuracy gap for this 2-class boundary (score 3 vs. 4). We will tune `maxIter` and `stepSize`.
- **XGBoost via SparkXGBClassifier**: Native L1/L2 regularization and efficient distributed training should further address the mild overfitting seen in Model 2.
- **Feature Engineering**: Adding TF-IDF vectors, sentence count, average word length, and punctuation density alongside Word2Vec to capture stylistic signals beyond token-level semantics.
- **Restore int_score=5**: Fix the fractions dict to include key `5` so the rare top-quality class is also considered in training, and evaluate whether a 3-class model can be meaningfully trained.


## Conclusion

**1. What is the conclusion of the 1st model?**

Our first model — a Random Forest classifier with `numTrees=5` and `maxDepth=2`, trained on 10-dimensional Word2Vec text embeddings and a scaled token-count feature — achieves **61.89% test accuracy** on what turned out to be an effective **2-class** educational quality scoring task (`int_score` 3 vs. 4). The raw dataset after filtering contains only three score values (3, 4, and 5); the stratified sampling step inadvertently drops `int_score=5` because it is absent from the fractions dict, leaving a binary classification problem. The train–test gap is negligible (0.3 pp), confirming that the shallow model generalizes well but is capacity-constrained: ~62% accuracy is only about 9.6 pp above the majority-class baseline of 52.3%. The pipeline design is validated — Word2Vec embeddings plus token-count carry real signal — but the current model is too shallow to exploit it fully.

**2. What can be done to possibly improve it?**

- **Increase model capacity**: As demonstrated by Model 2 (`numTrees=20`, `maxDepth=8`), deeper and more numerous trees raise test accuracy to **68.67%** (+6.78 pp) with only mild overfitting (1.31 pp train–test gap). Further increasing depth and trees — or switching to boosted ensembles — should continue to yield gains.
- **Fix the stratified sampling**: Add `5: 1.0` (or a suitable fraction) to the fractions dict so that `int_score=5` examples are retained, enabling a proper 3-class classification that matches the actual data distribution.
- **Richer text features**: Replacing or augmenting 10-dimensional Word2Vec with higher-dimensional vectors (50–100d), TF-IDF scores, sentence count, average word length, and punctuation density would provide richer representations of educational quality.
- **More training data**: The pipeline trains on only ~20% of the corpus (~96 K examples after splits). Scaling to a larger fraction would expose the model to more linguistic diversity and reduce variance.
- **Hyperparameter tuning**: A systematic `CrossValidator` + `ParamGridBuilder` sweep over `numTrees`, `maxDepth`, `minInstancesPerNode`, and Word2Vec `vectorSize` would find the optimal capacity without manual iteration.
- **Better embeddings**: Pre-trained sentence embeddings (e.g., GloVe, FastText, or projected BERT representations) would yield substantially richer text representations than a Word2Vec model trained on a filtered 20% sample.

**3. How did distributed computing help with this task?**

The FineWeb-Edu 10BT sample is composed of 14 Parquet files totalling billions of tokens — far too large to load into a single machine's memory. Running on SDSC Expanse with Spark (31 executor cores) enabled:
- **Parallel data loading and filtering** across 31 executor cores, reducing I/O bottlenecks while processing 1.9 M rows after cleaning the 20% sample
- **Distributed Word2Vec training** using partitioned gradient updates, completing in minutes instead of hours on so many documents
- **Parallel Random Forest construction** (Model 1 and Model 2 each took ~13 min 15 s) with trees built concurrently across executors
- **Scalable inference** via `.transform()` applied partition-by-partition — evaluating three splits (train 96 K / val 32 K / test 32 K) in just ~6 seconds for Model 1
- **Checkpointing** at two key stages (after cleaning and after stratified sampling) broke long lineage chains and prevented redundant recomputation during evaluation

Even the 20% sample used here (~1.9 M rows before stratified sampling) would challenge a local machine once Word2Vec and pipeline preprocessing are factored in. Training on the full 14-file dataset without distributed computing would be entirely infeasible.


## Save Best Model

Compare both models on the held-out test set and save the model with the best test accuracy to the `../models/` directory.

In [24]:
# Compare both models on the test set
results = [
    ("RF (numTrees=5,  maxDepth=2)",  model,  test_acc),
    ("RF (numTrees=20, maxDepth=8)",  model2, test_acc2),
]

best_name, best_model, best_acc = max(results, key=lambda x: x[2])
print(f"Best model    : {best_name}")
print(f"Test accuracy : {best_acc:.4f}")

# Persist the winner
model_dir = os.path.abspath(os.path.join("..", "models"))
os.makedirs(model_dir, exist_ok=True)

model_save_path = os.path.join(model_dir, "milestone3_best_rf_model")
# best_model.write().overwrite().save(model_save_path)
print(f"Success! Model saved to ../models/milestone3_best_rf_model")

Best model    : RF (numTrees=20, maxDepth=8)
Test accuracy : 0.6867
Success! Model saved to ../models/milestone3_best_rf_model
